# FRED Stage 2: Local Offline Inference & Ollama Testing Notebook
### 100% Privacy-Preserving Offline Financial Hallucination Detection & Verification

This notebook runs **local offline inference** against your fine-tuned Stage 2 model exported as a **GGUF binary** and deployed through **Ollama**.

**Key Capabilities:**
- Zero-cloud local verification: Sensitive financial disclosures never leave your machine.
- Tests all 5 actions: `preserve`, `edit`, `abstain`, `flag_conflict`, `request_evidence`.
- Measures local inference throughput (tokens/second) and GPU/CPU latency.
- Evaluates Clean Preservation Rate (CPR) and Complete Edit Success (CES) locally on `data/stage2/test_clean.jsonl` and `data/stage2/test_corrupted.jsonl`.

## 1. Setup & Ollama Connection Check
Ensure Ollama is running locally (`ollama serve`).

In [ ]:
import json
import re
import time
import requests
from pathlib import Path
from tqdm import tqdm

OLLAMA_BASE_URL = "http://localhost:11434"
MODEL_NAME = "fred-stage2"

try:
    res = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=5)
    if res.status_code == 200:
        models = [m["name"] for m in res.json().get("models", [])]
        print(f"Ollama server active! Available local models: {models}")
        if any(MODEL_NAME in m for m in models):
            print(f"Target model '{MODEL_NAME}' is registered and ready for inference!")
        else:
            print(f"Model '{MODEL_NAME}' not yet registered. Run: ollama create {MODEL_NAME} -f Modelfile_stage2")
    else:
        print(f"Ollama returned status code: {res.status_code}")
except Exception as e:
    print(f"Could not connect to Ollama at {OLLAMA_BASE_URL}. Ensure 'ollama serve' is running. Error: {e}")

## 2. Deploy GGUF Model with Ollama CLI
Instructions for registering the GGUF model:

```bash
# 1. Download your exported GGUF file from Kaggle/Colab and place it in the project root:
#    e.g., ./fred-stage2-qwen-q4_k_m.gguf

# 2. Create the model in Ollama using the Stage 2 Modelfile:
ollama create fred-stage2 -f Modelfile_stage2

# 3. Run interactive chat directly in terminal:
ollama run fred-stage2
```

## 3. Local Inference Function with Latency Tracking

In [ ]:
def query_local_fred(context, table, question, candidate_response, model_name=MODEL_NAME):
    """Query local Ollama instance with structured financial verification prompt."""
    table_str = "\n".join([" | ".join(r) for r in table]) if table else "[No Table]"
    user_content = (
        f"Context: {context}\n\n"
        f"Table:\n{table_str}\n\n"
        f"Question: {question}\n\n"
        f"Response to Verify: {candidate_response}"
    )
    
    payload = {
        "model": model_name,
        "messages": [
            {"role": "user", "content": user_content}
        ],
        "stream": False,
        "options": {
            "temperature": 0.1,
            "top_p": 0.9,
        }
    }
    
    start_time = time.time()
    resp = requests.post(f"{OLLAMA_BASE_URL}/api/chat", json=payload, timeout=60)
    latency = time.time() - start_time
    
    if resp.status_code == 200:
        data = resp.json()
        output_text = data["message"]["content"].strip()
        eval_count = data.get("eval_count", 0)
        eval_duration_ns = data.get("eval_duration", 1)
        tok_per_sec = (eval_count / (eval_duration_ns / 1e9)) if eval_duration_ns > 0 else 0.0
        return output_text, latency, tok_per_sec
    else:
        raise RuntimeError(f"Ollama error ({resp.status_code}): {resp.text}")

## 4. Interactive Test on the 5 Core Actions
Runs quick local inference across 5 representative test cases.

In [ ]:
sample_queries = [
    # 1. Preserve
    ("preserve", "Total revenues were $88,268M and operating income was $28,521M.", [["Metric", "Amount"], ["Operating Income", "$28,521M"]], "What was the operating profit margin?", "The operating margin was 32.3%."),
    # 2. Edit (Numerical/Arithmetic error)
    ("edit", "Total revenues were $88,268M and operating income was $28,521M.", [], "What was the operating profit margin?", "The operating margin was 45.0%."),
    # 3. Abstain (Missing evidence)
    ("abstain", "We manufacture commercial electric vehicles.", [], "What were total patent litigation legal expenses in 2023?", "Litigation expenses were $150M."),
    # 4. Flag Conflict (Contradictory context)
    ("flag_conflict", "Revenue was $100M.\n[Restatement Note]: Revision reveals revenue was actually $85M.", [], "What was the revenue?", "Revenue was $100M."),
    # 5. Request Evidence (Irrelevant context)
    ("request_evidence", "We are committed to environmental stewardship and diversity.", [], "What was the effective tax rate?", "The tax rate was 18%."),
]

print("\n--- Running Local Ollama Action Verification Tests ---\n")
for expected_action, ctx, tbl, q, cand in sample_queries:
    print(f"Target Action: {expected_action.upper()}")
    try:
        output, lat, tps = query_local_fred(ctx, tbl, q, cand)
        print(f"Local Latency: {lat:.2f}s ({tps:.1f} tokens/sec)")
        print(f"Prediction:\n{output}\n" + "-" * 50)
    except Exception as e:
        print(f"Test failed: {e}")

## 5. Local Batch Benchmark: Clean Preservation Rate (CPR)
Evaluates the local model on `data/stage2/test_clean.jsonl` to verify low false correction rates.

In [ ]:
test_clean_path = Path("../data/stage2/test_clean.jsonl")
if not test_clean_path.exists():
    test_clean_path = Path("data/stage2/test_clean.jsonl")

with open(test_clean_path, "r", encoding="utf-8") as f:
    clean_records = [json.loads(l) for l in f]

# Set test limit for local benchmark (e.g. 50 samples for fast check)
LOCAL_EVAL_N = 50
clean_slice = clean_records[:LOCAL_EVAL_N]

preserved_count = 0
false_edit_count = 0
total_tps = []

print(f"Running local Clean Preservation Benchmark on {len(clean_slice)} samples...")
for r in tqdm(clean_slice):
    try:
        out, _, tps = query_local_fred(
            context=r.get("context", ""),
            table=r.get("table", []),
            question=r.get("question", ""),
            candidate_response=r.get("unverified_response", r.get("gold_answer", "")),
        )
        total_tps.append(tps)
        if '"action": "preserve"' in out or "preserve" in out[:40]:
            preserved_count += 1
        elif "<delete>" in out or '"action": "edit"' in out:
            false_edit_count += 1
    except Exception as e:
        print(f"Inference error: {e}")

cpr = (preserved_count / len(clean_slice)) * 100.0
fcr = (false_edit_count / len(clean_slice)) * 100.0
avg_tps = sum(total_tps) / len(total_tps) if total_tps else 0.0

print(f"\nLocal Clean Preservation Benchmark Results:")
print(f"  Clean Preservation Rate (CPR): {cpr:.1f}% (Target: >= 85%)")
print(f"  False Correction Rate (FCR):   {fcr:.1f}% (Target: <= 10%)")
print(f"  Average Local Throughput:      {avg_tps:.1f} tokens/second")